# Nicoya mesh cross-section at constant y

Vertical cross-section of the FE mesh at a fixed along-strike position `y = const`.
Shows block structure (blockleft / blockright), fault trace, and top boundary.

Run in the `fenics_pygmt` environment (meshio, numpy, matplotlib available).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection, PolyCollection

try:
    import meshio
except ImportError:
    meshio = None
    print("meshio not found — will use built-in Gmsh 2.2 parser")


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
mesh_path   = Path('/home/staff/chao/SSEinv/Nicoya/mesh/nicoyaCKden_une_all.msh')
result_dir  = Path('/home/staff/chao/SSEinv/Nicoya/mesh/')
output_path = result_dir / 'nicoya_y30km_slice.pdf'

# ── Slice position ─────────────────────────────────────────────────────────
y_km = 30.0
y0   = y_km * 1e3        # metres

# Tolerance for "vertex is on the plane" (metres).
# Mesh coords are in metres — 1 m is appropriate.
tol = 1.0

# ── Physical group tags (from $PhysicalNames in .msh) ─────────────────────
TAG_TOP        = 1
TAG_FAULT      = 7
TAG_BLOCKLEFT  = 8
TAG_BLOCKRIGHT = 9


In [ ]:
# ── Mesh reader (meshio preferred, built-in Gmsh 2.2 fallback) ────────────

EDGE_PAIRS = ((0,1),(0,2),(0,3),(1,2),(1,3),(2,3))

def read_with_meshio(path):
    mesh = meshio.read(path)
    pts  = np.asarray(mesh.points[:, :3], dtype=float)

    tets, tets_tags = [], []
    tris, tris_tags = [], []

    for i, block in enumerate(mesh.cells):
        tags = mesh.cell_data["gmsh:physical"][i]
        if block.type == 'tetra':
            tets.append(block.data)
            tets_tags.append(tags)
        elif block.type == 'triangle':
            tris.append(block.data)
            tris_tags.append(tags)

    tets      = np.vstack(tets).astype(np.int64)       if tets      else np.empty((0,4), np.int64)
    tets_tags = np.concatenate(tets_tags).astype(int)  if tets_tags else np.array([], int)
    tris      = np.vstack(tris).astype(np.int64)       if tris      else np.empty((0,3), np.int64)
    tris_tags = np.concatenate(tris_tags).astype(int)  if tris_tags else np.array([], int)

    return pts, tets, tets_tags, tris, tris_tags


def read_gmsh22_manual(path):
    nodes, raw_tets, raw_tets_tags = None, [], []
    raw_tris, raw_tris_tags = [], []

    with open(path, encoding='utf-8') as f:
        for line in f:
            tag = line.strip()
            if tag == '$Nodes':
                n = int(next(f).strip())
                nodes = np.empty((n, 3), float)
                for _ in range(n):
                    p = next(f).split()
                    nodes[int(p[0])-1] = [float(p[1]), float(p[2]), float(p[3])]
            elif tag == '$Elements':
                n_elem = int(next(f).strip())
                for _ in range(n_elem):
                    p = next(f).split()
                    etype, ntags = int(p[1]), int(p[2])
                    phys_tag = int(p[3])
                    conn = [int(v)-1 for v in p[3+ntags:]]
                    if etype == 4:   # tetra
                        raw_tets.append(conn)
                        raw_tets_tags.append(phys_tag)
                    elif etype == 2: # triangle
                        raw_tris.append(conn)
                        raw_tris_tags.append(phys_tag)

    tets      = np.array(raw_tets,      np.int64)  if raw_tets else np.empty((0,4), np.int64)
    tets_tags = np.array(raw_tets_tags, int)
    tris      = np.array(raw_tris,      np.int64)  if raw_tris else np.empty((0,3), np.int64)
    tris_tags = np.array(raw_tris_tags, int)

    return nodes, tets, tets_tags, tris, tris_tags


def read_mesh(path):
    if meshio is not None:
        try:
            return read_with_meshio(path), 'meshio'
        except Exception as exc:
            print(f"meshio failed ({exc}), falling back to manual parser")
    return read_gmsh22_manual(path), 'manual'


In [ ]:
(nodes, tets, tets_tags, tris, tris_tags), backend = read_mesh(mesh_path)
print(f"Backend  : {backend}")
print(f"Nodes    : {len(nodes):,}")
print(f"Tets     : {len(tets):,}  (tags: {np.unique(tets_tags)})")
print(f"Triangles: {len(tris):,}  (tags: {np.unique(tris_tags)})")
print(f"X range  : {nodes[:,0].min()/1e3:.0f} – {nodes[:,0].max()/1e3:.0f} km")
print(f"Y range  : {nodes[:,1].min()/1e3:.0f} – {nodes[:,1].max()/1e3:.0f} km")
print(f"Z range  : {nodes[:,2].min()/1e3:.0f} – {nodes[:,2].max()/1e3:.0f} km")


In [ ]:
def _dedup(pts, tol):
    """Remove near-duplicate 3D points."""
    keep = []
    for p in pts:
        if all(np.linalg.norm(p - k) > tol for k in keep):
            keep.append(p)
    return keep


def tet_y_plane_polygon(tet_xyz, y0, tol=1.0):
    """
    Compute the convex polygon (x, z in km) from tet ∩ {y = y0}.
    Returns an (N, 2) float array (N = 3 or 4), or None if no intersection.
    """
    dy = tet_xyz[:, 1] - y0
    if np.all(dy > tol) or np.all(dy < -tol):
        return None

    pts = []
    for i, j in EDGE_PAIRS:
        d0, d1 = dy[i], dy[j]
        if abs(d0) <= tol and abs(d1) <= tol:   # edge lies on plane
            pts.extend([tet_xyz[i], tet_xyz[j]])
            continue
        if abs(d0) <= tol:
            pts.append(tet_xyz[i])
            continue
        if abs(d1) <= tol:
            pts.append(tet_xyz[j])
            continue
        if d0 * d1 < 0.0:                        # edge crosses plane
            t = d0 / (d0 - d1)
            pts.append(tet_xyz[i] + t * (tet_xyz[j] - tet_xyz[i]))

    pts = _dedup(pts, tol)
    if len(pts) < 3:
        return None

    pts_arr = np.array(pts)
    xz = pts_arr[:, [0, 2]]                      # drop y
    ctr = xz.mean(axis=0)
    angles = np.arctan2(xz[:, 1] - ctr[1], xz[:, 0] - ctr[0])
    return xz[np.argsort(angles)] / 1e3          # km


def tri_y_plane_segment(tri_xyz, y0, tol=1.0):
    """
    Compute the line segment (two (x,z) points, km) from triangle ∩ {y = y0}.
    Returns (2, 2) array or None.
    """
    dy = tri_xyz[:, 1] - y0
    if np.all(dy > tol) or np.all(dy < -tol):
        return None

    edge_pairs_tri = ((0,1),(0,2),(1,2))
    pts = []
    for i, j in edge_pairs_tri:
        d0, d1 = dy[i], dy[j]
        if abs(d0) <= tol and abs(d1) <= tol:
            pts.extend([tri_xyz[i], tri_xyz[j]])
            continue
        if abs(d0) <= tol:
            pts.append(tri_xyz[i])
            continue
        if abs(d1) <= tol:
            pts.append(tri_xyz[j])
            continue
        if d0 * d1 < 0.0:
            t = d0 / (d0 - d1)
            pts.append(tri_xyz[i] + t * (tri_xyz[j] - tri_xyz[i]))

    pts = _dedup(pts, tol)
    if len(pts) < 2:
        return None
    pts_arr = np.array(pts[:2])                  # take first two unique points
    return pts_arr[:, [0, 2]] / 1e3              # km


In [ ]:
# ── Vectorised pre-filter: tets that straddle y = y0 ─────────────────────
y_verts = nodes[tets, 1]                          # (N_tets, 4)
straddle = (y_verts.min(axis=1) <= y0) & (y_verts.max(axis=1) >= y0)
tets_cut  = tets[straddle]
tags_cut  = tets_tags[straddle]
print(f"Tets straddling y = {y_km:.0f} km: {straddle.sum():,} / {len(tets):,}")

# ── Compute intersection polygons per block ────────────────────────────────
polys_left  = []   # blockleft  (tag 8)
polys_right = []   # blockright (tag 9)

for tet_idx, tag in zip(tets_cut, tags_cut):
    poly = tet_y_plane_polygon(nodes[tet_idx], y0, tol)
    if poly is None:
        continue
    if tag == TAG_BLOCKLEFT:
        polys_left.append(poly)
    elif tag == TAG_BLOCKRIGHT:
        polys_right.append(poly)

print(f"  blockleft  polygons : {len(polys_left):,}")
print(f"  blockright polygons : {len(polys_right):,}")

# ── Fault trace (tag 7 triangles) ─────────────────────────────────────────
fault_tris = tris[tris_tags == TAG_FAULT]
y_ftri = nodes[fault_tris, 1]
fstraddle = (y_ftri.min(axis=1) <= y0) & (y_ftri.max(axis=1) >= y0)
fault_segs = []
for tri_idx in fault_tris[fstraddle]:
    seg = tri_y_plane_segment(nodes[tri_idx], y0, tol)
    if seg is not None:
        fault_segs.append(seg)
print(f"Fault trace segments : {len(fault_segs):,}")

# ── Top boundary trace (tag 1 triangles) ──────────────────────────────────
top_tris = tris[tris_tags == TAG_TOP]
y_ttri = nodes[top_tris, 1]
tstraddle = (y_ttri.min(axis=1) <= y0) & (y_ttri.max(axis=1) >= y0)
top_segs = []
for tri_idx in top_tris[tstraddle]:
    seg = tri_y_plane_segment(nodes[tri_idx], y0, tol)
    if seg is not None:
        top_segs.append(seg)
print(f"Top boundary segments: {len(top_segs):,}")


In [ ]:
# ── Plot styling ───────────────────────────────────────────────────────────
COLOR_LEFT  = '#4878CF'   # blue  – blockleft
COLOR_RIGHT = '#D65F5F'   # red   – blockright
COLOR_FAULT = 'darkgray'         # black – fault trace
COLOR_TOP   = 'k'   # dark grey – top boundary
ALPHA_FILL  = 0.55
EDGE_LW     = 0.5        # mesh edge linewidth
EDGE_COLOR  = '0'

# ── Axis limits (km) — set None to auto ───────────────────────────────────
X_LIM = (-300, 300)    # along-dip  (x)
Z_LIM = (-200, 5)      # depth      (z)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5), dpi=300)

# ── Filled polygons (block structure) ─────────────────────────────────────
if polys_left:
    pc_left  = PolyCollection(polys_left,  facecolor=COLOR_LEFT,  alpha=ALPHA_FILL,
                               edgecolor=EDGE_COLOR, linewidth=EDGE_LW)
    ax.add_collection(pc_left)

if polys_right:
    pc_right = PolyCollection(polys_right, facecolor=COLOR_RIGHT, alpha=ALPHA_FILL,
                               edgecolor=EDGE_COLOR, linewidth=EDGE_LW)
    ax.add_collection(pc_right)

# ── Top boundary ──────────────────────────────────────────────────────────
if top_segs:
    lc_top = LineCollection(top_segs, colors=COLOR_TOP, linewidth=1.5,
                             linestyle='-', label='Top boundary')
    ax.add_collection(lc_top)

# ── Fault trace ───────────────────────────────────────────────────────────
if fault_segs:
    lc_fault = LineCollection(fault_segs, colors=COLOR_FAULT, linewidth=2,
                               label='Fault')
    ax.add_collection(lc_fault)

# ── Axes ──────────────────────────────────────────────────────────────────
if X_LIM:
    ax.set_xlim(X_LIM)
else:
    ax.autoscale_view()

if Z_LIM:
    ax.set_ylim(Z_LIM)

ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Along-dip (km)', fontsize=11)
ax.set_ylabel('Depth (km)',     fontsize=11)
ax.set_title(f'Mesh cross-section at y = {y_km:g} km (along-strike)', fontsize=12)

# Depth increases downward  → do NOT invert y (z is already negative)

# ── Legend ────────────────────────────────────────────────────────────────
legend_handles = [
    mpatches.Patch(facecolor=COLOR_LEFT,  alpha=ALPHA_FILL, label='Cocos Plate'),
    mpatches.Patch(facecolor=COLOR_RIGHT, alpha=ALPHA_FILL, label='Caribbean Plate'),
    plt.Line2D([0], [0], color=COLOR_FAULT, lw=2, label='Plate interface'),
    plt.Line2D([0], [0], color=COLOR_TOP,   lw=1.2, linestyle='--', label='Top boundary'),
]
ax.legend(handles=legend_handles, loc='lower right', fontsize=9, framealpha=0.85)

ax.grid(True, linestyle=':', linewidth=0.4, alpha=0.5)
fig.tight_layout()
plt.show()


In [ ]:
fig.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Saved to {output_path}')


In [ ]:
# ── PyGMT version — helpers ───────────────────────────────────────────────
try:
    import pygmt
    HAS_PYGMT = True
except ImportError:
    HAS_PYGMT = False
    print('pygmt not available — skipping PyGMT plot')


def write_gmt_polys(polys, path, color_hex, alpha=0):
    """
    Write polygons to a GMT multi-segment file with per-segment fill.
    alpha: 0–100 transparency (0 = opaque).  Uses GMT @transparency suffix.
    """
    fill = f'{color_hex}@{alpha}' if alpha > 0 else color_hex
    with open(path, 'w') as f:
        for poly in polys:
            f.write(f'> -G{fill}\n')
            closed = np.vstack([poly, poly[0]])
            for x, z in closed:
                f.write(f'{x:.6f} {z:.6f}\n')


def write_gmt_lines(segs, path):
    """Write line segments to a GMT multi-segment file (no fill)."""
    with open(path, 'w') as f:
        for seg in segs:
            f.write('>\n')
            for x, z in seg:
                f.write(f'{x:.6f} {z:.6f}\n')


In [ ]:
if not HAS_PYGMT:
    print('Skipping PyGMT plot (pygmt not available)')
else:

    # Equal-aspect panel: 1 km = 1 km on the page
    x_range_km   = X_LIM[1] - X_LIM[0]
    z_range_km   = Z_LIM[1] - Z_LIM[0]
    panel_width  = 14.0
    panel_height = panel_width * z_range_km / x_range_km

    region = [X_LIM[0], X_LIM[1], Z_LIM[0], Z_LIM[1]]
    proj   = f'X{panel_width}c/{panel_height:.3f}c'

    fig_gmt = pygmt.Figure()

    pygmt.config(
        MAP_FRAME_TYPE      = 'plain',
        FONT_ANNOT_PRIMARY  = '6p,Helvetica,black',
        FONT_LABEL          = '8p',
        MAP_FRAME_PEN       = '0.5p,black',
    )

    fig_gmt.basemap(
        region=region, projection=proj,
        frame=['WSne',
               'xa100f50+lAlong-dip (km)',
               'ya50f25+lDepth (km)'],
    )

    # ── Temporary segment files ────────────────────────────────────────────
    _tmp = {k: f'/tmp/gmt_mesh_{k}.txt'
            for k in ('left', 'right', 'fault', 'top')}

    if polys_left:
        write_gmt_polys(polys_left,  _tmp['left'],  '#4878CF', alpha=45)
        fig_gmt.plot(data=_tmp['left'],  pen='0.1p,black')

    if polys_right:
        write_gmt_polys(polys_right, _tmp['right'], '#D65F5F', alpha=45)
        fig_gmt.plot(data=_tmp['right'], pen='0.1p,black')

    if top_segs:
        write_gmt_lines(top_segs, _tmp['top'])
        fig_gmt.plot(data=_tmp['top'], pen='1.2p,black')

    if fault_segs:
        write_gmt_lines(fault_segs, _tmp['fault'])
        fig_gmt.plot(data=_tmp['fault'], pen='2p,darkgray')

    # ── Legend (spec must be written to a file — PyGMT requires a path) ───
    _leg_file = '/tmp/gmt_mesh_legend.txt'
    with open(_leg_file, 'w') as _f:
        _f.write(
            'S 0.3c r 0.3c #4878CF 0.5p,white 0.8c Cocos Plate\n'
            'S 0.3c r 0.3c #D65F5F 0.5p,white 0.8c Caribbean Plate\n'
            'S 0.3c - 0.5c - 2p,darkgray 0.8c Plate interface\n'
            'S 0.3c - 0.5c - 1.2p,black 0.8c Top boundary\n'
        )
    fig_gmt.legend(spec=_leg_file,
                   position='JBR+jBR+o0.2c',
                   box='+gwhite+p0.5p,gray50')

    # ── Slice label ───────────────────────────────────────────────────────
    fig_gmt.text(
        x=X_LIM[1] - 5,  y=Z_LIM[1] - 3,
        text=f'Along-strike = {y_km:g} km',
        font="8p,Helvetica-Bold,white", justify="TR",
        fill="gray60", offset="0.1c/0.05c",
    )

    fig_gmt.show()


In [ ]:
# Save PyGMT figure
output_path_gmt = result_dir / f'nicoya_y{y_km:.0f}km_slice_pygmt.pdf'
if HAS_PYGMT:
    fig_gmt.savefig(str(output_path_gmt), dpi=300)
    print(f'Saved to {output_path_gmt}')


In [ ]:
# ── Build PyVista UnstructuredGrid from the already-loaded tet mesh ─────────
import pyvista as pv
import matplotlib.colors as mcolors

# Assemble tet connectivity in PyVista format: [4, v0, v1, v2, v3, ...]
n_tets   = len(tets)
cells_pv = np.hstack([np.full((n_tets, 1), 4, dtype=np.int64), tets]).ravel()
types_pv = np.full(n_tets, pv.CellType.TETRA, dtype=np.uint8)

grid_pv = pv.UnstructuredGrid(cells_pv, types_pv, nodes)
grid_pv.cell_data["block"] = tets_tags.astype(float)

# Also build surface grids for fault and top boundary
def make_surface_pv(tri_indices, nodes):
    n = len(tri_indices)
    cells_t = np.hstack([np.full((n, 1), 3, dtype=np.int64), tri_indices]).ravel()
    types_t = np.full(n, pv.CellType.TRIANGLE, dtype=np.uint8)
    return pv.UnstructuredGrid(cells_t, types_t, nodes)

fault_pv = make_surface_pv(tris[tris_tags == TAG_FAULT],  nodes)
top_pv   = make_surface_pv(tris[tris_tags == TAG_TOP],    nodes)

print(grid_pv)
print(f"Fault triangles : {fault_pv.n_cells:,}")
print(f"Top   triangles : {top_pv.n_cells:,}")


In [ ]:
# ── PyVista slice at y = y0, normal = (0, 1, 0) ─────────────────────────────
import os, warnings
import matplotlib.colors as mcolors
from IPython.display import Image, display

origin = [0.0, y0, 0.0]
normal = [0.0, 1.0, 0.0]

grid_pt      = grid_pv.cell_data_to_point_data()
slice_blocks = grid_pt.slice(normal=normal, origin=origin)
slice_fault  = fault_pv.slice(normal=normal, origin=origin)
slice_top    =   top_pv.slice(normal=normal, origin=origin)

print(f"Slice: {slice_blocks.n_points:,} points, {slice_blocks.n_cells:,} cells")

cmap_blocks = mcolors.ListedColormap(["#4878CF", "#D65F5F"])

# off_screen=True uses VTK EGL/osmesa — no display needed
plotter = pv.Plotter(window_size=[1800, 700], off_screen=True)
plotter.set_background("white")
plotter.add_mesh(slice_blocks,
                 scalars="block", cmap=cmap_blocks, clim=[7.5, 9.5],
                 show_edges=True, edge_color="black", line_width=0.3,
                 opacity=0.7, show_scalar_bar=False)
if slice_fault.n_points > 0:
    plotter.add_mesh(slice_fault, color="dimgray", line_width=2.5)
if slice_top.n_points > 0:
    plotter.add_mesh(slice_top,   color="black",   line_width=1.5)

plotter.view_xz()
plotter.add_title(f"PyVista slice  y = {y_km:g} km (along-strike)", font_size=10)

img = plotter.screenshot(return_img=True)   # renders off-screen → numpy array
plotter.close()

# Save and display inline
output_pv = result_dir / f"nicoya_y{y_km:.0f}km_slice_pyvista.png"
import PIL.Image
PIL.Image.fromarray(img).save(output_pv)
display(Image(filename=str(output_pv)))
print(f"Saved to {output_pv}")


In [ ]:
# ── PyVista slice geometry + matplotlib PolyCollection rendering ─────────────
# PyVista computes the intersection (VTK vtkCutter); matplotlib renders the result.
# This avoids any display/OpenGL dependency while still using PyVista's general
# slice (arbitrary normal vector, not just axis-aligned).

origin = [0.0, y0, 0.0]
normal = [0.0, 1.0, 0.0]

grid_pt      = grid_pv.cell_data_to_point_data()
slice_blocks = grid_pt.slice(normal=normal, origin=origin)

print(f"Slice: {slice_blocks.n_points:,} points, {slice_blocks.n_cells:,} cells")
print(f"Faces array size: {len(slice_blocks.faces):,}  (mixed tri/quad polygons)")

# ── Parse mixed tri/quad face array → polygon lists ───────────────────────
pts_km     = slice_blocks.points[:, [0, 2]] / 1e3   # (x, z) in km
block_vals = slice_blocks["block"]

faces_flat = np.asarray(slice_blocks.faces)
polys_left_pv, polys_right_pv = [], []

i = 0
while i < len(faces_flat):
    n    = int(faces_flat[i])
    vids = faces_flat[i+1 : i+1+n]
    poly_xz  = pts_km[vids]
    mean_tag = block_vals[vids].mean()
    if mean_tag < 8.5:
        polys_left_pv.append(poly_xz)
    else:
        polys_right_pv.append(poly_xz)
    i += n + 1

print(f"  blockleft  polygons: {len(polys_left_pv):,}")
print(f"  blockright polygons: {len(polys_right_pv):,}")

# ── Render with PolyCollection ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5), dpi=150)

if polys_left_pv:
    ax.add_collection(PolyCollection(polys_left_pv,  facecolor="#4878CF",
                                     alpha=ALPHA_FILL, edgecolor=EDGE_COLOR,
                                     linewidth=EDGE_LW))
if polys_right_pv:
    ax.add_collection(PolyCollection(polys_right_pv, facecolor="#D65F5F",
                                     alpha=ALPHA_FILL, edgecolor=EDGE_COLOR,
                                     linewidth=EDGE_LW))

# Fault and top boundary from manual intersection (cells 5–6)
if fault_segs:
    ax.add_collection(LineCollection(fault_segs, colors="dimgray",
                                     linewidth=2.0, label="Plate interface"))
if top_segs:
    ax.add_collection(LineCollection(top_segs,   colors="black",
                                     linewidth=1.5, label="Top boundary"))

ax.set_xlim(X_LIM);  ax.set_ylim(Z_LIM)
ax.set_aspect("equal")
ax.set_xlabel("Along-dip (km)");  ax.set_ylabel("Depth (km)")
ax.set_title(f"PyVista slice  y = {y_km:g} km (along-strike)")

legend_handles = [
    mpatches.Patch(facecolor="#4878CF", alpha=ALPHA_FILL, label="Cocos Plate"),
    mpatches.Patch(facecolor="#D65F5F", alpha=ALPHA_FILL, label="Caribbean Plate"),
    plt.Line2D([0],[0], color="dimgray", lw=2,   label="Plate interface"),
    plt.Line2D([0],[0], color="black",   lw=1.5, label="Top boundary"),
]
ax.legend(handles=legend_handles, loc="lower right", fontsize=9, framealpha=0.85)
ax.grid(True, ls=":", lw=0.4, alpha=0.5)
fig.tight_layout()
plt.show()


In [ ]:
# ── PyVista geometry + PyGMT rendering (same style as cells 10–12) ───────────
if not HAS_PYGMT:
    print("pygmt not available — skipping")
else:
    # Re-use slice_blocks from cell 15; re-compute fault/top slices from PyVista
    slice_fault_pv = fault_pv.slice(normal=normal, origin=origin)
    slice_top_pv   =   top_pv.slice(normal=normal, origin=origin)

    # ── Extract fault/top line segments from PyVista .lines ─────────────────
    def _pv_lines_to_segs(poly):
        """Parse PyVista PolyData .lines [n,i,j,...] → list of (2,2) km arrays."""
        if poly.n_points == 0 or len(poly.lines) == 0:
            return []
        pts_xz = poly.points[:, [0, 2]] / 1e3
        flat = np.asarray(poly.lines)
        segs, i = [], 0
        while i < len(flat):
            n    = int(flat[i])
            vids = flat[i+1 : i+1+n]
            for k in range(len(vids) - 1):
                segs.append(pts_xz[[vids[k], vids[k+1]]])
            i += n + 1
        return segs

    fault_segs_pv = _pv_lines_to_segs(slice_fault_pv)
    top_segs_pv   = _pv_lines_to_segs(slice_top_pv)

    # ── PyGMT figure ─────────────────────────────────────────────────────────
    x_range_km  = X_LIM[1] - X_LIM[0]
    z_range_km  = Z_LIM[1] - Z_LIM[0]
    panel_width = 14.0
    panel_height = panel_width * z_range_km / x_range_km

    region = [X_LIM[0], X_LIM[1], Z_LIM[0], Z_LIM[1]]
    proj   = f"X{panel_width}c/{panel_height:.3f}c"

    fig_pv = pygmt.Figure()
    pygmt.config(MAP_FRAME_TYPE="plain", FONT_ANNOT_PRIMARY="6p,Helvetica,black",
                 FONT_LABEL="8p", MAP_FRAME_PEN="0.5p,black")
    fig_pv.basemap(region=region, projection=proj,
                   frame=["WSne",
                          "xa100f50+lAlong-dip (km)",
                          "ya50f25+lDepth (km)"])

    _tmp2 = {k: f"/tmp/gmt_pv_{k}.txt"
             for k in ("left", "right", "fault", "top")}

    if polys_left_pv:
        write_gmt_polys(polys_left_pv,  _tmp2["left"],  "#4878CF", alpha=45)
        fig_pv.plot(data=_tmp2["left"],  pen="0.1p,black")
    if polys_right_pv:
        write_gmt_polys(polys_right_pv, _tmp2["right"], "#D65F5F", alpha=45)
        fig_pv.plot(data=_tmp2["right"], pen="0.1p,black")
    if top_segs_pv:
        write_gmt_lines(top_segs_pv,   _tmp2["top"])
        fig_pv.plot(data=_tmp2["top"],   pen="1.2p,black")
    if fault_segs_pv:
        write_gmt_lines(fault_segs_pv, _tmp2["fault"])
        fig_pv.plot(data=_tmp2["fault"], pen="2p,darkgray")

    _leg_file2 = "/tmp/gmt_pv_legend.txt"
    with open(_leg_file2, "w") as _f:
        _f.write(
            "S 0.3c r 0.3c #4878CF@45 0.5p,white 0.8c Cocos Plate\n"
            "S 0.3c r 0.3c #D65F5F@45 0.5p,white 0.8c Caribbean Plate\n"
            "S 0.3c - 0.5c - 2p,darkgray 0.8c Plate interface\n"
            "S 0.3c - 0.5c - 1.2p,black 0.8c Top boundary\n"
        )
    fig_pv.legend(spec=_leg_file2, position="JBR+jBR+o0.2c",
                  box="+gwhite+p0.5p,gray50")
    fig_pv.text(x=X_LIM[1]-5, y=Z_LIM[1]-3,
                text=f"Along-strike = {y_km:g} km",
                font="8p,Helvetica-Bold,white", justify="TR",
                fill="gray60", offset="0.1c/0.05c")
    fig_pv.show()

    output_pv_gmt = result_dir / f"nicoya_y{y_km:.0f}km_slice_pyvista_pygmt.pdf"
    fig_pv.savefig(str(output_pv_gmt))
    print(f"Saved to {output_pv_gmt}")


In [ ]:
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.collections as mc

# Load the mesh
mesh = pv.read('/home/staff/chao/SSEinv/Nicoya/mesh/nicoyaCKden_une_all.msh')

# Define the slice plane: origin and normal vector
# Example: along-strike = 30 km means y = 30000 m (adjust units to match your mesh)
origin = (0, 30000, 0)   # a point on the plane
normal = (0, 1, 0)       # normal vector pointing in along-strike direction (y-axis)

# Perform the slice
sliced = mesh.slice(normal=normal, origin=origin)

In [ ]:
# sliced is a PolyData object; extract the edges
edges = sliced.extract_all_edges()

# Get points and lines connectivity
points = edges.points          # shape (N, 3): x, y, z coordinates
lines  = edges.lines           # flat array: [2, i0, i1, 2, i2, i3, ...]

# Parse the lines array into segments
segments = []
i = 0
while i < len(lines):
    n = lines[i]               # should always be 2 for line segments
    idx0, idx1 = lines[i+1], lines[i+2]
    p0 = points[idx0]
    p1 = points[idx1]
    segments.append([p0, p1])
    i += n + 1

In [ ]:
# Decide which two coordinates to plot
# For a y-slice, x-axis = along-dip (x), y-axis = depth (z)
# Convert from meters to km if needed
scale = 1e-3   # m -> km

seg_plot = [
    [[p0[0]*scale, p0[2]*scale],
     [p1[0]*scale, p1[2]*scale]]
    for p0, p1 in segments
]

fig, ax = plt.subplots(figsize=(10, 4))

lc = mc.LineCollection(seg_plot, linewidths=0.4, colors='black', alpha=0.7)
ax.add_collection(lc)

# Set axis ranges explicitly — full control here
ax.set_xlim(-300, 300)    # along-dip range in km
ax.set_ylim(-200, 0)      # depth range in km (negative = down)

ax.set_xlabel("Along-dip (km)")
ax.set_ylabel("Depth (km)")
ax.set_aspect('equal')
ax.set_title(f"Along-strike = 30 km")

plt.tight_layout()
# plt.savefig("mesh_cross_section.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from paraview.simple import *
s = Slice()
props = s.ListProperties()
for p in props:
    print(p)

In [ ]:
import subprocess
import os

# Write the pvpython script to a temp file and execute it
pvpython_script = """

from paraview.simple import *
from pathlib import Path

# ── User parameters ─────────────────────────────────────────────────────────
MESH_FILE    = "/home/staff/chao/SSEinv/Nicoya/rst_locking/mu_true_nicoyaCKden_une_all_mul40u40.xdmf"
OUTPUT_PDF   = "test_meshCS.pdf"

# Slice plane in mesh units (meters)
SLICE_ORIGIN = [0.0, 30000.0, 0.0]   # along-strike = 30 km → y = 30000 m
SLICE_NORMAL = [0.0, 1.0,     0.0]   # normal in along-strike (y) direction

# Axis display ranges in km (after Calculator rescale)
XRANGE = [-300, 300]    # along-dip (km)
ZRANGE = [-200,   5]    # depth (km)

# Figure size in pixels
FIG_WIDTH_PX  = 4000
FIG_HEIGHT_PX = 1200
# ────────────────────────────────────────────────────────────────────────────

paraview.simple._DisableFirstRenderCameraReset()

# 1. Load the XDMF mesh
reader = OpenDataFile(MESH_FILE)
UpdatePipeline()

# 2. Rescale coordinates from meters to km via Calculator
calc = Calculator(Input=reader)
calc.Function          = "coords/1000"
calc.CoordinateResults = 1
calc.ResultArrayName   = "coords_km"
UpdatePipeline()

# 3. Apply Slice filter
slicer = Slice(Input=calc)
slicer.SliceType            = 'Plane'
slicer.SliceType.Origin     = [o/1000 for o in SLICE_ORIGIN]   # convert to km
slicer.SliceType.Normal     = SLICE_NORMAL
slicer.Triangulatetheslice  = 0
slicer.PointMergeMethod     = "Uniform Binning"
slicer.Crinkleslice         = 0
UpdatePipeline()

# 4. Set up render view
view = CreateRenderView()
view.ViewSize                     = [FIG_WIDTH_PX, FIG_HEIGHT_PX]
view.UseColorPaletteForBackground = 0
view.Background                   = [1.0, 1.0, 1.0]   # dark gray
view.InteractionMode              = '2D'
view.CameraParallelProjection     = 1

# 5. Display slice as Surface With Edges
display = Show(slicer, view)
display.Representation = 'Surface With Edges'
display.LineWidth       = 0.5
display.Opacity         = 1.0
display.Specular        = 0.0

# Must call ColorBy first, then set colors — otherwise array coloring overrides
ColorBy(display, None)
display.SetScalarBarVisibility(view, False)
display.AmbientColor   = [0.9, 0.9, 0.8]   # light gray faces
display.DiffuseColor   = [0.9, 0.9, 0.8]
display.EdgeColor      = [0.0, 0.0, 0.0]   # black edges

# 6. Configure orthographic camera
# X horizontal (along-dip), Z vertical (depth), looking along -Y
cx = (XRANGE[0] + XRANGE[1]) / 2.0   #   0 km
cz = (ZRANGE[0] + ZRANGE[1]) / 2.0   # -100 km

view.CameraPosition      = [cx, SLICE_ORIGIN[1]/1000 - 1e6, cz]   # behind slice on -Y side
view.CameraFocalPoint    = [cx, SLICE_ORIGIN[1]/1000,        cz]   # looking toward +Y
view.CameraViewUp        = [0.0, 0.0, 1.0]                        # Z is up

# Parallel scale = half the height of the desired view extent
view.CameraParallelScale = (ZRANGE[1] - ZRANGE[0]) / 2.0          # 100 km

# 7. Axes grid — white labels, km units
view.AxesGrid.Visibility        = 1
view.AxesGrid.XTitle            = 'Along-dip (km)'
view.AxesGrid.YTitle            = ''
view.AxesGrid.ZTitle            = 'Depth (km)'
view.AxesGrid.XTitleFontSize    = 14
view.AxesGrid.ZTitleFontSize    = 14
view.AxesGrid.XLabelFontSize    = 15
view.AxesGrid.ZLabelFontSize    = 15
view.AxesGrid.XTitleColor       = [0.0, 0.0, 0.0]   # white
view.AxesGrid.ZTitleColor       = [0.0, 0.0, 0.0]
view.AxesGrid.XLabelColor       = [0.0, 0.0, 0.0]
view.AxesGrid.ZLabelColor       = [0.0, 0.0, 0.0]
view.AxesGrid.GridColor         = [0.0, 0.0, 0.0]
view.AxesGrid.ShowGrid          = 0
view.AxesGrid.ShowEdges         = 1


Render()

# 8. Export as PDF
ExportView(OUTPUT_PDF, view=view, Compressoutputfile=0)
# SaveScreenshot(OUTPUT_PDF.replace('.pdf', '.png'), view,
#                ImageResolution=[FIG_WIDTH_PX, FIG_HEIGHT_PX],
#                OverrideColorPalette='')
print(f"Saved: {OUTPUT_PDF}")

"""

# Write script to file
script_path = "run_paraview_slice.py"
with open(script_path, "w") as f:
    f.write(pvpython_script)

# Run with pvpython
result = subprocess.run(
    ["pvpython", script_path],
    capture_output=True,
    text=True
)

print("STDOUT:", result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

os.remove(script_path)